# Notebook 6 — Train Model LightGBM

**Mục tiêu:** Dự đoán doanh thu phim (`revenue`) bằng LightGBM với ba chiến lược biến đổi target (`log`, `sqrt`, `raw`) và blend prediction.

**Input:** `train_fe.csv`, `test_fe.csv`

## 1. Import thư viện

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm

import joblib


## 2. Load dữ liệu

Notebook ưu tiên dùng `train_fe_v2.csv` và `test_fe_v2.csv` vì đây là dữ liệu đã được bổ sung các feature như:

- `log_budget`
- `log_runtime`
- `budget_per_runtime`
- `log_budget_per_runtime`
- `genre_count`
- `release_decade`
- `movie_age`

Nếu chưa có dữ liệu v2, notebook sẽ tự tạo nhanh các feature này từ `train_fe.csv` và `test_fe.csv`.


### Feature bổ sung

Hàm `add_quick_features` tạo thêm các feature dựa trên biến từ NB5:

- **`log_budget`, `log_runtime`**: log-transform giúp nén phân phối lệch phải, đồng nhất scale với target.
- **`budget_per_runtime`**: mật độ đầu tư sản xuất theo phút phim.
- **`genre_count`, `movie_age`, `release_decade`**: bổ sung bối cảnh thể loại và thời gian.

In [ ]:
def add_quick_features(df, ref_max_year=None):
    df = df.copy()

    if "budget" in df.columns and "log_budget" not in df.columns:
        df["log_budget"] = np.log1p(df["budget"].clip(lower=0))

    if "runtime" in df.columns and "log_runtime" not in df.columns:
        df["log_runtime"] = np.log1p(df["runtime"].clip(lower=0))

    if "budget" in df.columns and "runtime" in df.columns and "budget_per_runtime" not in df.columns:
        runtime_safe = df["runtime"].replace(0, np.nan)
        df["budget_per_runtime"] = df["budget"] / (runtime_safe + 1)
        df["budget_per_runtime"] = (
            df["budget_per_runtime"]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
        )
        df["log_budget_per_runtime"] = np.log1p(df["budget_per_runtime"].clip(lower=0))

    genre_cols = [col for col in df.columns if col.startswith("genre_")]
    if len(genre_cols) > 0 and "genre_count" not in df.columns:
        df["genre_count"] = df[genre_cols].sum(axis=1)

    if "release_year" in df.columns:
        if "release_decade" not in df.columns:
            df["release_decade"] = (df["release_year"] // 10) * 10

        if "movie_age" not in df.columns:
            max_year = ref_max_year if ref_max_year is not None else df["release_year"].max()
            df["movie_age"] = max_year - df["release_year"]

    if "has_budget" in df.columns and "budget_missing" not in df.columns:
        df["budget_missing"] = df["has_budget"]

    return df


In [ ]:
if os.path.exists("../data/processed/train_fe_v2.csv") and os.path.exists("../data/processed/test_fe_v2.csv"):
    train = pd.read_csv("../data/processed/train_fe_v2.csv")
    test = pd.read_csv("../data/processed/test_fe_v2.csv")
    print("Đang dùng train_fe_v2.csv và test_fe_v2.csv")
else:
    train = pd.read_csv("../data/processed/train_fe.csv")
    test  = pd.read_csv("../data/processed/test_fe.csv")

    max_year = train["release_year"].max() if "release_year" in train.columns else None
    train = add_quick_features(train, max_year)
    test  = add_quick_features(test,  max_year)

    print("Chưa có file v2, đã tạo feature nhanh từ train_fe.csv và test_fe.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())

## 3. Kiểm tra dữ liệu đầu vào

Trước khi train, nhóm kiểm tra nhanh:

- Kích thước dữ liệu
- Missing values
- Dòng trùng
- Phân phối của target `revenue`


In [ ]:
print("Missing values trong train:")
display(train.isnull().sum()[train.isnull().sum() > 0])

print("Missing values trong test:")
display(test.isnull().sum()[test.isnull().sum() > 0])

print("Số dòng trùng trong train:", train.duplicated().sum())
print("Số dòng trùng trong test:", test.duplicated().sum())

print("Thống kê revenue trong train:")
display(train["revenue"].describe())


## 4. Phân phối target

`revenue` thường bị lệch phải rất mạnh. Một số phim có doanh thu rất cao, trong khi nhiều phim có doanh thu thấp hơn rất nhiều.

Vì vậy notebook này so sánh nhiều cách biến đổi target:

- `log1p(revenue)`: tốt cho RMSLE và R² log.
- `sqrt(revenue)`: trung gian giữa log scale và raw scale, giúp cải thiện R² raw.
- `revenue`: train trực tiếp trên doanh thu gốc, tập trung hơn vào raw scale.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(train["revenue"], bins=50)
plt.xlabel("Revenue")
plt.ylabel("Số lượng phim")
plt.title("Phân phối revenue")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(np.log1p(train["revenue"]), bins=50)
plt.xlabel("log1p(revenue)")
plt.ylabel("Số lượng phim")
plt.title("Phân phối log1p(revenue)")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(np.sqrt(train["revenue"]), bins=50)
plt.xlabel("sqrt(revenue)")
plt.ylabel("Số lượng phim")
plt.title("Phân phối sqrt(revenue)")
plt.grid(alpha=0.3)
plt.show()


## 5. Gộp dữ liệu và chia lại train/test

Gộp `train_fe` và `test_fe` sau feature engineering, lọc `revenue >= 1`, sau đó chia lại 80/20 với `stratify` theo nhóm `log1p(revenue)` để phân phối target cân bằng hơn giữa hai tập.

In [ ]:
# Gộp train và test sau feature engineering, lọc revenue >= 1
data = pd.concat([train, test], ignore_index=True)
data = data[data["revenue"] >= 1].reset_index(drop=True)

print(f"Tổng số mẫu sau khi gộp: {data.shape[0]}")
display(data[["budget", "revenue", "runtime", "director_encoded"]].describe().round(2))

## 6. Bổ sung feature interaction

Bản final thêm một số interaction feature để giúp model nhận diện tốt hơn các phim có khả năng doanh thu cao.

Các feature được thêm:

- `director_x_log_budget`: kết hợp sức ảnh hưởng của đạo diễn và ngân sách.
- `log_budget_x_runtime`: kết hợp ngân sách và thời lượng phim.
- `budget_per_genre`: ngân sách trung bình theo số lượng thể loại.

Các feature này được thêm nhanh trong file train để không phải sửa lại toàn bộ pipeline cũ.


In [ ]:
if "director_encoded" in data.columns and "log_budget" in data.columns:
    data["director_x_log_budget"] = data["director_encoded"] * data["log_budget"]

if "log_budget" in data.columns and "log_runtime" in data.columns:
    data["log_budget_x_runtime"] = data["log_budget"] * data["log_runtime"]

if "budget" in data.columns and "genre_count" in data.columns:
    data["budget_per_genre"] = data["budget"] / (data["genre_count"] + 1)
    data["budget_per_genre"] = data["budget_per_genre"].replace([np.inf, -np.inf], np.nan).fillna(0)

new_interaction_cols = [
    col for col in ["director_x_log_budget", "log_budget_x_runtime", "budget_per_genre"]
    if col in data.columns
]

display(data[new_interaction_cols].describe().T)

## 7. Chuẩn bị feature và target

Notebook tạo 3 target khác nhau:

- `y_log = log1p(revenue)`
- `y_sqrt = sqrt(revenue)`
- `y_raw = revenue`

Trong đó:

- `log_model` tối ưu tốt trên log scale.
- `sqrt_model` thường cân bằng hơn giữa log scale và raw scale.
- `raw_model` tập trung hơn vào doanh thu gốc.


In [ ]:
target_col = "revenue"

X = data.drop(columns=[target_col]).copy()
y_raw = data[target_col].copy()
y_log = np.log1p(y_raw)
y_sqrt = np.sqrt(y_raw)

X = X.replace([np.inf, -np.inf], np.nan)
median_values = X.median(numeric_only=True)
X = X.fillna(median_values)

print("Số dòng:", X.shape[0])
print("Số feature:", X.shape[1])
print("Missing còn lại:", X.isnull().sum().sum())


## 8. Chia train/test

Dữ liệu được chia lại theo tỉ lệ 80/20.

Notebook dùng `stratify` theo nhóm `log1p(revenue)` để đảm bảo tập train và test có phân phối target tương đối giống nhau.


In [ ]:
target_bins = pd.qcut(y_log, q=10, duplicates="drop")

X_train, X_test, y_train_raw, y_test_raw, y_train_log, y_test_log, y_train_sqrt, y_test_sqrt = train_test_split(
    X,
    y_raw,
    y_log,
    y_sqrt,
    test_size=0.2,
    random_state=42,
    stratify=target_bins
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print()
print(f"y_train_raw mean: {y_train_raw.mean():,.2f}")
print(f"y_test_raw mean : {y_test_raw.mean():,.2f}")
print(f"y_train_log mean: {y_train_log.mean():.4f}")
print(f"y_test_log mean : {y_test_log.mean():.4f}")


## 9. Hàm đánh giá

Notebook đánh giá model trên cả hai thang đo:

### Revenue/raw scale

- `R2_raw`
- `RMSE_raw`
- `MAE_raw`

### Log scale

- `RMSLE`
- `R2_log`

Ngoài ra có thêm `SMAPE` để xem sai số phần trăm đối xứng.


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(np.where(denominator == 0, 0, np.abs(y_true - y_pred) / denominator)) * 100


def eval_pred(y_true_raw, pred_raw, model_name):
    pred_raw = np.clip(pred_raw, 0, None)
    y_true_log = np.log1p(y_true_raw)
    pred_log = np.log1p(pred_raw)

    return {
        "model": model_name,
        "R2_raw": r2_score(y_true_raw, pred_raw),
        "RMSE_raw": rmse(y_true_raw, pred_raw),
        "MAE_raw": mean_absolute_error(y_true_raw, pred_raw),
        "RMSLE": rmse(y_true_log, pred_log),
        "R2_log": r2_score(y_true_log, pred_log),
        "SMAPE": smape(y_true_raw, pred_raw)
    }


## 10. Baseline model

Trước khi train LightGBM, nhóm dùng `DummyRegressor` làm baseline đơn giản. Model này chỉ dự đoán giá trị trung bình, nên dùng để kiểm tra LightGBM có thật sự học được tín hiệu từ dữ liệu hay không.


In [ ]:
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train_raw)

dummy_pred = dummy_model.predict(X_test)
dummy_score = pd.DataFrame([eval_pred(y_test_raw, dummy_pred, "dummy_mean")])

display(dummy_score)


## 11. Train LightGBM với 3 loại target

Notebook dùng cùng một bộ tham số LightGBM cho 3 model:

1. `log_model`: train trên `log1p(revenue)`.
2. `raw_model`: train trực tiếp trên `revenue`.
3. `sqrt_model`: train trên `sqrt(revenue)`.

Trong các thử nghiệm trước, `sqrt_model` là hướng giúp R² raw vượt 60% vì nó không nén doanh thu cao quá mạnh như log target, nhưng vẫn giảm độ lệch tốt hơn raw target.


### Các phiên bản mô hình

Nhóm huấn luyện bốn phiên bản LightGBM với cùng bộ tham số, khác nhau ở biến mục tiêu:

| Mô hình | Biến mục tiêu | Mục đích |
|---|---|---|
| **`raw_model`** | `revenue` | Tối ưu trực tiếp trên thang doanh thu gốc |
| **`log_model`** | `log1p(revenue)` | Giảm ảnh hưởng phân phối lệch phải; tốt trên thang log |
| **`sqrt_model`** | `sqrt(revenue)` | Cân bằng giữa raw và log — không nén doanh thu cao mạnh như log |
| **`blend_model`** | Tổ hợp dự đoán | Kết hợp theo trọng số tối ưu từ ba model trên |

In [ ]:
params = {
    "n_estimators": 300,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 5,
    "min_child_samples": 30,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.5,
    "reg_lambda": 1.0
}

predictions = {}
summary_rows = []

# 1. Log target model
log_model = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=1,
    verbosity=-1,
    **params
)
log_model.fit(X_train, y_train_log)

pred_log = np.expm1(log_model.predict(X_test))
predictions["log_model"] = np.clip(pred_log, 0, None)
summary_rows.append(eval_pred(y_test_raw, predictions["log_model"], "log_model"))


# 2. Raw target model
raw_model = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=1,
    verbosity=-1,
    **params
)
raw_model.fit(X_train, y_train_raw)

pred_raw = raw_model.predict(X_test)
predictions["raw_model"] = np.clip(pred_raw, 0, None)
summary_rows.append(eval_pred(y_test_raw, predictions["raw_model"], "raw_model"))


# 3. Sqrt target model
sqrt_model = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=1,
    verbosity=-1,
    **params
)
sqrt_model.fit(X_train, y_train_sqrt)

pred_sqrt = sqrt_model.predict(X_test) ** 2
predictions["sqrt_model"] = np.clip(pred_sqrt, 0, None)
summary_rows.append(eval_pred(y_test_raw, predictions["sqrt_model"], "sqrt_model"))

model_summary = pd.DataFrame(summary_rows).sort_values("R2_raw", ascending=False).reset_index(drop=True)
display(model_summary)


## 12. Cross-validation nhanh cho `sqrt_model`

Vì `sqrt_model` là model chính để cải thiện R² raw, notebook kiểm tra thêm bằng 5-Fold Cross Validation trên train set.

Metric chính trong phần này là `R2_raw` sau khi chuyển dự đoán từ sqrt scale về revenue gốc.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []

for fold, (train_idx, valid_idx) in enumerate(tqdm(cv.split(X_train), total=5, desc="CV sqrt_model"), start=1):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_valid = X_train.iloc[valid_idx]
    y_fold_train_sqrt = y_train_sqrt.iloc[train_idx]
    y_fold_valid_raw = y_train_raw.iloc[valid_idx]

    fold_model = LGBMRegressor(
        objective="regression",
        random_state=42,
        n_jobs=1,
        verbosity=-1,
        **params
    )
    fold_model.fit(X_fold_train, y_fold_train_sqrt)

    fold_pred_raw = fold_model.predict(X_fold_valid) ** 2
    fold_pred_raw = np.clip(fold_pred_raw, 0, None)

    cv_rows.append(eval_pred(y_fold_valid_raw, fold_pred_raw, f"fold_{fold}"))

cv_report = pd.DataFrame(cv_rows)
display(cv_report)

print("Mean CV R2_raw:", cv_report["R2_raw"].mean())
print("Std CV R2_raw :", cv_report["R2_raw"].std())
print("Mean CV RMSLE :", cv_report["RMSLE"].mean())


## 13. Blend prediction

Sau khi có 3 model, notebook thử blend prediction để xem có tăng thêm R² raw hay không.

Cách blend:

```python
prediction = w_sqrt * sqrt_model + w_raw * raw_model + w_log * log_model
```

Notebook thử các trọng số cách nhau 0.05 và chọn tổ hợp có R² raw cao nhất.


In [ ]:
blend_rows = []

for w_sqrt in np.arange(0, 1.01, 0.05):
    for w_raw in np.arange(0, 1.01 - w_sqrt, 0.05):
        w_log = 1 - w_sqrt - w_raw

        pred_blend = (
            w_sqrt * predictions["sqrt_model"]
            + w_raw * predictions["raw_model"]
            + w_log * predictions["log_model"]
        )

        row = eval_pred(y_test_raw, pred_blend, "blend_model")
        row["weight_sqrt_model"] = round(float(w_sqrt), 2)
        row["weight_raw_model"] = round(float(w_raw), 2)
        row["weight_log_model"] = round(float(w_log), 2)
        blend_rows.append(row)

blend_result = pd.DataFrame(blend_rows).sort_values("R2_raw", ascending=False).reset_index(drop=True)

display(blend_result.head(10))

best_blend = blend_result.iloc[0]

pred_blend = (
    best_blend["weight_sqrt_model"] * predictions["sqrt_model"]
    + best_blend["weight_raw_model"] * predictions["raw_model"]
    + best_blend["weight_log_model"] * predictions["log_model"]
)

blend_score = eval_pred(y_test_raw, pred_blend, "blend_model")
final_summary = pd.concat([
    dummy_score,
    model_summary,
    pd.DataFrame([blend_score])
], ignore_index=True).sort_values("R2_raw", ascending=False).reset_index(drop=True)

display(final_summary)


## Chọn model chính


In [ ]:
best_model_name = final_summary.iloc[0]["model"]

if best_model_name == "blend_model":
    best_pred = pred_blend
elif best_model_name == "sqrt_model":
    best_pred = predictions["sqrt_model"]
elif best_model_name == "raw_model":
    best_pred = predictions["raw_model"]
elif best_model_name == "log_model":
    best_pred = predictions["log_model"]
else:
    best_pred = dummy_pred

print("Model tốt nhất theo R2_raw:", best_model_name)
print("R2_raw:", r2_score(y_test_raw, best_pred))
print("RMSE_raw:", rmse(y_test_raw, best_pred))
print("MAE_raw:", mean_absolute_error(y_test_raw, best_pred))
print("RMSLE:", rmse(np.log1p(y_test_raw), np.log1p(np.clip(best_pred, 0, None))))


### Đánh giá kết quả — Phân tích tổng hợp

Kết quả thực nghiệm cho thấy **`blend_model`** đạt hiệu quả tốt nhất trên thang doanh thu gốc với **R² ≈ 0.648**, RMSE ≈ 75.5 triệu USD và MAE ≈ 15.7 triệu USD. Điều này có nghĩa mô hình giải thích được khoảng **64.8%** phương sai của doanh thu phim trong tập kiểm tra.

**So sánh chi tiết các phiên bản mô hình:**

- **`log_model`**: cho kết quả tốt nhất trên **thang logarit** (R² log ≈ 0.91, RMSLE ≈ 1.44), do được tối ưu trực tiếp trên `log1p(revenue)`. Tuy nhiên khi quy đổi về thang doanh thu gốc, R² raw thấp hơn (≈ 0.50) vì log đã nén quá mạnh các giá trị doanh thu cao — mô hình có xu hướng underestimate blockbuster.

- **`sqrt_model`**: cân bằng tốt hơn giữa log scale và raw scale (R² raw ≈ 0.645). Không nén doanh thu cao mạnh như log, nên R² raw cải thiện rõ rệt so với log_model.

- **`raw_model`**: train trực tiếp trên doanh thu gốc (R² raw ≈ 0.628) nhưng dễ bị chi phối bởi các phim bom tấn doanh thu rất cao, dẫn đến RMSE lớn hơn và khả năng tổng quát kém hơn.

- **`blend_model`**: kết hợp dự đoán từ cả ba model theo trọng số tối ưu (sqrt ≈ 0.75, raw ≈ 0.25), tận dụng ưu điểm của từng cách biểu diễn target. Đây là model được chọn làm **kết quả cuối cùng** vì đạt R² raw cao nhất.

**Nhận xét chung:**  
Mô hình LightGBM với pipeline tiền xử lý và kỹ thuật đặc trưng đã xây dựng đạt kết quả tốt trên bài toán dự đoán doanh thu phim với chỉ metadata trước phát hành. Kết quả không nên được diễn giải như một con số dự đoán tuyệt đối — mô hình phù hợp hơn với vai trò hỗ trợ phân tích xu hướng và ước lượng khung doanh thu, đặc biệt khi bối cảnh thiếu thông tin quảng bá và phản ứng khán giả.

## 15. Visualize Actual vs Predicted

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test_raw, best_pred, alpha=0.35)
limit_value = np.percentile(np.concatenate([y_test_raw.values, best_pred]), 95)
plt.xlim(0, limit_value)
plt.ylim(0, limit_value)
plt.plot([0, limit_value], [0, limit_value], linestyle="--")
plt.xlabel("Actual revenue")
plt.ylabel("Predicted revenue")
plt.title(f"Actual vs Predicted - {best_model_name}")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(np.log1p(y_test_raw), np.log1p(np.clip(best_pred, 0, None)), alpha=0.35)
min_value = min(np.log1p(y_test_raw).min(), np.log1p(np.clip(best_pred, 0, None)).min())
max_value = max(np.log1p(y_test_raw).max(), np.log1p(np.clip(best_pred, 0, None)).max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
plt.xlabel("Actual log1p(revenue)")
plt.ylabel("Predicted log1p(revenue)")
r2_log_val = r2_score(np.log1p(y_test_raw), np.log1p(np.clip(best_pred, 0, None)))
plt.title(f"Actual vs Predicted - Log scale  R² log = {r2_log_val:.4f}")
plt.grid(alpha=0.3)
plt.show()

### Nhận xét: Actual vs Predicted

Trên **thang log**, các điểm bám tương đối tốt theo đường chéo ở vùng doanh thu trung bình. Hai đầu phân phối (doanh thu rất thấp hoặc rất cao) có xu hướng bị kéo về vùng trung bình — đây là hiện tượng **regression-to-the-mean** phổ biến khi target lệch phải mạnh. Blockbuster phụ thuộc vào nhiều yếu tố ngoài metadata (marketing, franchise, phản ứng khán giả) nên mô hình khó dự đoán chính xác.

## 16. Residual analysis

In [ ]:
residual_raw = y_test_raw - best_pred
residual_log = np.log1p(y_test_raw) - np.log1p(np.clip(best_pred, 0, None))

plt.figure(figsize=(8, 5))
plt.scatter(best_pred, residual_raw, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted revenue")
plt.ylabel("Residual raw")
plt.title("Residual plot trên revenue scale")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(residual_log, bins=40)
plt.xlabel("Residual log")
plt.ylabel("Số lượng phim")
plt.title("Phân phối residual trên log scale")
plt.grid(alpha=0.3)
plt.show()


### Nhận xét: Residual

Residual trên **raw scale** phân tán tăng dần theo predicted value — dấu hiệu của **heteroscedasticity**: mô hình ổn định ở phim doanh thu thấp/trung bình nhưng sai số lớn hơn ở blockbuster.

Residual trên **log scale** phân phối gần đối xứng quanh 0, xác nhận log-transform giúp chuẩn hóa sai số tốt hơn.

## 17. Error analysis theo nhóm revenue

In [ ]:
prediction = X_test.copy()
prediction["actual_revenue"] = y_test_raw.values
prediction["pred_log_model"] = predictions["log_model"]
prediction["pred_raw_model"] = predictions["raw_model"]
prediction["pred_sqrt_model"] = predictions["sqrt_model"]
prediction["pred_blend_model"] = pred_blend
prediction["pred_best_model"] = best_pred
prediction["abs_error_best"] = np.abs(prediction["actual_revenue"] - prediction["pred_best_model"])
prediction["abs_error_log_best"] = np.abs(np.log1p(prediction["actual_revenue"]) - np.log1p(np.clip(prediction["pred_best_model"], 0, None)))

prediction["revenue_group"] = pd.qcut(
    prediction["actual_revenue"],
    q=5,
    duplicates="drop"
)

error_by_group = prediction.groupby("revenue_group").agg(
    count=("actual_revenue", "count"),
    mean_actual_revenue=("actual_revenue", "mean"),
    mean_abs_error=("abs_error_best", "mean"),
    mean_abs_error_log=("abs_error_log_best", "mean")
).reset_index()

display(error_by_group)


## 18. Feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance_log_model": log_model.feature_importances_,
    "importance_raw_model": raw_model.feature_importances_,
    "importance_sqrt_model": sqrt_model.feature_importances_
})

feature_importance["importance_avg"] = feature_importance[
    ["importance_log_model", "importance_raw_model", "importance_sqrt_model"]
].mean(axis=1)

feature_importance = feature_importance.sort_values("importance_avg", ascending=False)

display(feature_importance.head(20))

top_features = feature_importance.head(15).sort_values("importance_avg")

plt.figure(figsize=(8, 6))
plt.barh(top_features["feature"], top_features["importance_avg"])
plt.xlabel("Average importance")
plt.ylabel("Feature")
plt.title("Top 15 feature importance")
plt.grid(axis="x", alpha=0.3)
plt.show()


### Nhận xét: Feature Importance

Ba nhóm tín hiệu chính:

- **Đạo diễn & ngân sách**: `director_encoded` là feature quan trọng nhất — lịch sử doanh thu đạo diễn là proxy mạnh cho uy tín thương mại. `budget` và `log_budget` cũng đóng góp cao.
- **Interaction features**: `log_budget_x_runtime`, `director_x_log_budget` và `budget_per_runtime` nằm trong top 10, cho thấy mô hình khai thác được quan hệ phi tuyến mà các biến đơn lẻ không nắm bắt được.
- **Thời gian & thể loại**: `runtime`, `release_year`, `month_sin/cos` và một số genre flags đóng góp nhỏ hơn nhưng vẫn có mặt trong top 15.

## 19. Lưu kết quả

In [ ]:
final_summary.to_csv("../data/processed/lightgbm_final_model_summary.csv", index=False)
cv_report.to_csv("../data/processed/lightgbm_sqrt_model_cv_report.csv", index=False)
blend_result.to_csv("../data/processed/lightgbm_blend_result.csv", index=False)
prediction.to_csv("../data/processed/lightgbm_prediction_final.csv", index=False)
feature_importance.to_csv("../data/processed/lightgbm_feature_importance_final.csv", index=False)
error_by_group.to_csv("../data/processed/lightgbm_error_by_group_final.csv", index=False)

joblib.dump(
    {
        "log_model": log_model,
        "raw_model": raw_model,
        "sqrt_model": sqrt_model,
        "best_model_name": best_model_name,
        "best_blend": best_blend.to_dict(),
        "feature_cols": list(X.columns),
        "median_values": median_values
    },
    "../data/processed/lightgbm_movie_revenue_final.pkl"
)

print("Đã lưu:")
print("- ../data/processed/lightgbm_final_model_summary.csv")
print("- ../data/processed/lightgbm_sqrt_model_cv_report.csv")
print("- ../data/processed/lightgbm_blend_result.csv")
print("- ../data/processed/lightgbm_prediction_final.csv")
print("- ../data/processed/lightgbm_feature_importance_final.csv")
print("- ../data/processed/lightgbm_error_by_group_final.csv")
print("- ../data/processed/lightgbm_movie_revenue_final.pkl")

## 20. Nhận xét kết quả

- **`blend_model`** (sqrt ≈ 0.75, raw ≈ 0.25) đạt R² raw cao nhất ≈ **0.648**, RMSE ≈ 75.5M USD, MAE ≈ 15.7M USD.
- `sqrt_model` cân bằng tốt giữa log và raw scale; `log_model` tốt nhất trên log scale (RMSLE ≈ 1.44) nhưng thấp hơn về R² raw; `raw_model` dễ bị ảnh hưởng bởi outlier doanh thu cao.
- Metric phù hợp cho bài toán regression lệch phải: **R² raw**, **RMSLE**, **MAE/RMSE raw**.